# 02 · Wikipedia — LLM-built knowledge graph + Text2Cypher

An LLM extracted a typed knowledge graph from Wikipedia into AgensGraph. This
notebook tours the graph and asks questions in natural language — the LLM writes
the Cypher.

> Run `build.py` first to build the `wikipedia_kg` graph.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

configure_settings()  # Settings.llm / Settings.embed_model
store = agens.make_pg_store("wikipedia_kg", vector_dimension=EMBED_DIM,
                            enhanced_schema=True, create=False)
llm = get_llm()

## The extracted graph

Entity counts by type, and a few example relationships.

In [2]:
import pandas as pd
# An element is written on the label naming what it is, so its type is label(n).
# This unwound an n.labels list, which no longer exists.
comp = store.structured_query("""MATCH (n:"__Node__") WITH label(n) AS t
    WHERE t <> 'Chunk'
    RETURN t AS type, count(*) AS n ORDER BY n DESC""")
display(pd.DataFrame(comp))
rels = store.structured_query("""MATCH (a:"__Node__")-[r]->(b:"__Node__")
    WHERE label(a) <> 'Chunk' AND label(b) <> 'Chunk'
    RETURN a.name AS source, type(r) AS rel, b.name AS target LIMIT 8""")
pd.DataFrame(rels)


,type,n
0,WORK,35
1,PLACE,28
2,EVENT,24
3,PERSON,22
4,ORGANIZATION,18
5,MOVEMENT,1


,source,rel,target
0,Anarchism,INFLUENCED,Libertarian Socialism
1,Surface albedo,PART_OF,Albedo
2,Albedo,CREATED,Surface albedo
3,Albedo,INFLUENCED,Ice–albedo feedback
4,Albedo,MEMBER_OF,solar radiation
5,Ice–albedo feedback,SUCCEEDED,climate change
6,Albedo,FOUNDED,reflectance
7,Albedo,PART_OF,solar energy spectrum


## The schema fed to Text2Cypher

`enhanced_schema=True` adds example values, helping the LLM write good Cypher.

In [3]:
print(store.get_schema_str()[:1200])

Node properties:
Chunk {_node_content: STRING, _node_type: STRING, doc_id: STRING, document_id: STRING, embedding: LIST, id: STRING (e.g. 09f227cb-cbd9-4798-96dc-bc4124b36139, 345472b1-233c-4dac-803e-c0a3f8c1ad03, 67c49f7a-e985-408a-aff6-aad494e2832d, 7a4eae08-542b-48a4-af12-4620f44e4546, 91184c0a-2ae6-416c-b1ec-e81d7a87eb0a), ref_doc_id: STRING, text: STRING (e.g. Abraham Lincoln ( ; February 12, 1809 – April 15, 1865) was an American lawyer, politician, and statesman who served as the 16th president of the United States from 1861 until his assassination in 1865. Lincoln led the Union through the American Civil War to defend the nation as a constitutional union and succeeded in defeating the insurgent Confederacy, abolishing slavery, expanding the power of the federal government, and modernizing the U.S. economy.

Lincoln was born into poverty in a log cabin in Kentucky and was raised on the frontier, primarily in Indiana. He was self-educated and became a lawyer, Whig Party leader, I

## Text2Cypher — natural language → AgensGraph Cypher

The store ships an AgensGraph-dialect prompt, so a plain `TextToCypherRetriever`
generates runnable Cypher. `SafeTextToCypherRetriever` (in `_common/cypher.py`)
adds error-isolation so a bad generation can't crash the query.

In [4]:
from llama_index_agensgraph.retrievers import (
    SafeTextToCypherRetriever,
    strip_markdown,
)

# What the generated statement may do is the server's decision: it runs in a
# transaction that cannot write. That is not a boundary against a role which may
# run a command on the server's host, and a superuser may -- which is what this
# notebook connects as, to a database it owns. Accepted here deliberately.
t2c = SafeTextToCypherRetriever(
    graph_store=store,
    llm=llm,
    cypher_validator=strip_markdown,
    allow_server_programs=True,
)
nodes = t2c.retrieve("How many entities of each type are there?")
print(nodes[0].node.text if nodes else "(no result)")


Generated Cypher query:
MATCH (n:"__Node__")
RETURN label(n) AS type, count(*) AS n ORDER BY n DESC LIMIT 50

Cypher Response:
[{'type': 'WORK', 'n': 35}, {'type': 'PLACE', 'n': 28}, {'type': 'EVENT', 'n': 24}, {'type': 'PERSON', 'n': 22}, {'type': 'ORGANIZATION', 'n': 18}, {'type': 'Chunk', 'n': 12}, {'type': 'MOVEMENT', 'n': 1}]


## Full retriever stack

Combine keyword (`LLMSynonymRetriever`), vector (`VectorContextRetriever`) and
Text2Cypher retrieval behind one query engine.

In [5]:
from llama_index.core import PropertyGraphIndex
from llama_index.core.indices.property_graph import (
    LLMSynonymRetriever,
    VectorContextRetriever,
)

index = PropertyGraphIndex.from_existing(
    store, embed_model=get_embed_model(), llm=llm, kg_extractors=[], use_async=False
)
qe = index.as_query_engine(
    sub_retrievers=[
        LLMSynonymRetriever(graph_store=store, llm=llm, include_text=True),
        VectorContextRetriever(
            graph_store=store,
            embed_model=get_embed_model(),
            similarity_top_k=5,
            path_depth=1,
            include_text=True,
        ),
        SafeTextToCypherRetriever(
            graph_store=store,
            llm=llm,
            cypher_validator=strip_markdown,
            allow_server_programs=True,
        ),
    ]
)
print(qe.query("What is anarchism, and what people or movements is it connected to?"))


Anarchism is a political philosophy and movement that questions all forms of authority and seeks to eliminate institutions that it views as coercive and hierarchical, such as nation-states and capitalism. It advocates for stateless societies and voluntary associations. Anarchism is historically associated with the left-wing political spectrum, particularly as a part of libertarian socialism.

The anarchist movement has connections to various historical events and struggles, particularly in the 19th and 20th centuries, where it played a significant role in workers' movements for emancipation. Anarchists have participated in notable events such as the Paris Commune, the Russian Civil War, and the Spanish Civil War. Additionally, the movement has influenced and been influenced by various individuals and other movements, particularly within the realms of anti-capitalism, anti-war, and anti-globalization efforts in more recent decades.


## How it was built

`build.py` extracts the graph with one call:

```python
extractor = SchemaLLMPathExtractor(llm=llm,
    possible_entities=Literal["Person","Organization","Place","Event","Work"],
    possible_relations=Literal["FOUNDED","LOCATED_IN","BORN_IN", ...], strict=False)
PropertyGraphIndex.from_documents(docs, property_graph_store=store,
    kg_extractors=[extractor], embed_model=embed, llm=llm)
```

In [6]:
agens.close()